# Object-embedding batch processing

Embed every `*_semantics.json` file recursively under a configurable list of
folders using [core.dom.embedding.object](01.core.dom.embedding.object.ipynb).
This is the embedding companion to [the object-summary batch notebook](05.application.summary.object.ipynb).

Each document's nonempty `summary` and all recursive object `summary` fields become
vectors in a dedicated Chroma collection. SQLite retains raw content, source IDs,
types, paths, section context, and parent/child links. Mirrored records are indexed
once. Empty or intentionally ignored summaries remain in the catalog without vectors.
Section/prose `s` fields are not additional vector records.

Run cells in order. Opening this notebook does not process corpora. Source JSON,
Markdown, summaries, and summary checkpoints are never modified. Live processing
cells are excluded from automated notebook tests.

## Configuration

Set `INPUT_DIRS` to folders containing existing `*_semantics.json` files; nested
folders are included. Defaults match the outputs of the object-summary application.
Folders are processed in list order, with sorted files within each folder. Overlapping
roots and repeated paths are deduplicated globally; the first folder owns the file.
Missing or empty folders fail discovery before the index is opened.

Optional configuration is loaded from `PROJ_ROOT/.env`, with existing environment
variables taking precedence. `OLLAMA_EMBED_MODEL` defaults to `qwen3-embedding:8b`,
independent of the summarization model. `OLLAMA_EMBED_HOSTS` defaults to the
comma-separated pair `http://localhost:11434,http://172.27.74.16:11434`.
Both services must have the same embedding model installed. `OLLAMA_API_KEY`
supplies optional bearer authentication. `RIBOSOME_EMBED_REQUEST_TIMEOUT` defaults
to 300 seconds including queue wait. `RIBOSOME_EMBED_SERVER_CONCURRENCY` defaults
to four requests per host (eight concurrent requests in total). Faster hosts take more
work as slots become available. Configuration and discovery make no provider
requests and do not create index files.

`INDEX_DIR` defaults to `PROJ_ROOT/db/object_embeddings/qwen3-embedding-8b`. Keep host set, model, and index location consistent when resuming.
Model-specific directories preserve prior `bge-m3` indexes. A provider-set change
requires a separate index directory; changing concurrency alone does not.
`PREVIEW_OUTPUTS` limits inspection only, never
processing. Leave `FORCE_REBUILD=False` to resume. Set it to `True` before running
configuration to select a fresh timestamped directory and recompute all embeddings.
This preserves earlier indexes. Repeated runs against that same new directory resume;
rerun configuration with the flag enabled to select another fresh directory.
`CATALOG_PATH` is updated together with `INDEX_DIR`. No summary checkpoints are reset.

In [ ]:
# | notest
import asyncio
import json
import math
import os
import sys
from dataclasses import asdict
from datetime import datetime
from pathlib import Path
from time import perf_counter

from dotenv import load_dotenv


def resolve_project_root():
    configured = os.getenv("PROJ_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "pyproject.toml").is_file():
            raise FileNotFoundError(f"Invalid PROJ_ROOT (missing pyproject.toml): {root}")
        return root
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        if (root / "pyproject.toml").is_file():
            return root
    raise FileNotFoundError("Cannot locate PROJ_ROOT containing pyproject.toml")


PROJ_ROOT = resolve_project_root()
load_dotenv(PROJ_ROOT / ".env", override=False)
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

import pandas as pd
from IPython.display import display
from utils.embedding_ollama_pool import OllamaEmbeddingPool
from utils.embedding_progress import EmbeddingProgressDisplay
from tqdm.auto import tqdm
from ribosome.core.dom.embedding.object import ObjectEmbeddingIndex

INPUT_DIRS = [
    PROJ_ROOT / "assets" / "code-docs.4.0" / ".md",
    PROJ_ROOT / "assets" / "code-docs.5.0" / ".md",
]
EMBED_MODEL = (os.getenv("OLLAMA_EMBED_MODEL") or "qwen3-embedding:8b").strip()
OLLAMA_HOSTS = list(dict.fromkeys(host.strip().rstrip("/") for host in os.getenv(
    "OLLAMA_EMBED_HOSTS", "http://localhost:11434,http://172.27.74.16:11434"
).split(",") if host.strip()))
SERVER_CONCURRENCY = int(os.getenv("RIBOSOME_EMBED_SERVER_CONCURRENCY", "4"))
EMBEDDING_CONCURRENCY = len(OLLAMA_HOSTS) * SERVER_CONCURRENCY
INDEX_DIR = PROJ_ROOT / "db" / "object_embeddings" / EMBED_MODEL.replace("/", "_").replace(":", "-")
FORCE_REBUILD = False  # True creates a fresh index; the previous index is preserved.
if FORCE_REBUILD:
    INDEX_DIR = INDEX_DIR.with_name(f"{INDEX_DIR.name}-{datetime.now():%Y%m%d-%H%M%S-%f}")
CATALOG_PATH = INDEX_DIR / "catalog.sqlite3"
REQUEST_TIMEOUT = float(os.getenv("RIBOSOME_EMBED_REQUEST_TIMEOUT", "300"))
if not OLLAMA_HOSTS or not EMBED_MODEL or SERVER_CONCURRENCY < 1:
    raise ValueError("Embedding hosts/model must be nonempty and server concurrency must be positive")
if not math.isfinite(REQUEST_TIMEOUT) or REQUEST_TIMEOUT <= 0:
    raise ValueError("RIBOSOME_EMBED_REQUEST_TIMEOUT must be finite and positive")
EMBEDDING_CONFIG = {"provider": "ollama-pool", "hosts": sorted(OLLAMA_HOSTS)}
api_key = os.getenv("OLLAMA_API_KEY")
headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
PREVIEW_OUTPUTS = 3
PROGRESS_INTERVAL = 0.25
print("Inputs:")
for input_dir in INPUT_DIRS:
    print(f"  {input_dir}")
print(f"Embedding hosts: {OLLAMA_HOSTS}\nEmbedding model: {EMBED_MODEL}\nConcurrent requests: {EMBEDDING_CONCURRENCY}")
print(f"Index directory: {INDEX_DIR}\nCatalog: {CATALOG_PATH}")

## Discovery and batch helpers

The document bar shows completed files, the current source path, status counts, and
elapsed time. It refreshes while awaiting the embedding provider. One file is published
at a time, respecting the catalog's single-writer contract. Unique summaries within
each file run concurrently through bounded async workers on both Ollama servers. Individual failures are
reported and other files continue. Cancelling stops and awaits the active task, leaves
prior published generations intact, and retains collected results for inspection.
Synchronous parsing and Chroma writes finish before cancellation can be observed.
The concurrency panel refreshes every `PROGRESS_INTERVAL` seconds (default 250 ms).
It shows running and queued HTTP requests, colored per-server activity against the
four-request limit, outcome counts, oldest request age, current-file elapsed time,
and completed files. Document failures and request failures are labeled separately.
A final refresh follows cancellation cleanup. Parsing, cached-vector lookups and
catalog publication can continue when no HTTP requests are running.


In [ ]:
import asyncio
from dataclasses import asdict
from pathlib import Path
from time import perf_counter

from tqdm.auto import tqdm


def discover_semantics(input_dirs):
    """Validate every folder and deduplicate resolved files in configured order."""
    if isinstance(input_dirs, (str, Path)):
        input_dirs = [input_dirs]
    input_dirs = list(input_dirs)
    if not input_dirs:
        raise ValueError("INPUT_DIRS must contain at least one folder")
    corpora, seen = [], set()
    for value in input_dirs:
        folder = Path(value).expanduser().resolve()
        if not folder.is_dir():
            raise FileNotFoundError(f"Semantics input directory does not exist: {folder}")
        files = sorted({path.resolve() for path in folder.rglob("*_semantics.json") if path.is_file()})
        if not files:
            raise ValueError(f"No *_semantics.json files found under: {folder}")
        unique = [path for path in files if path not in seen]
        seen.update(unique)
        corpora.append((folder, unique))
    return corpora


async def embed_corpora(corpora, object_index, results, *, progress=True,
                        progress_interval=0.25, status_display=None):
    """Append finished-file reports and update optional live status; await cancellation cleanup."""
    if not 0 < progress_interval < float("inf"):
        raise ValueError("progress_interval must be finite and positive")
    counts = {"indexed": 0, "unchanged": 0, "failed": 0}
    total = sum(len(files) for _, files in corpora)
    state, current_file = "Running", ""

    def refresh_status():
        if status_display is not None:
            status_display.update(current_file=current_file, report=results, state=state)

    with tqdm(total=total, desc="Documents", unit="file", disable=not progress) as bar:
        try:
            for folder, files in corpora:
                for file in files:
                    current_file = file
                    started = perf_counter()
                    _ = bar.set_postfix(file=str(file), **counts)
                    refresh_status()
                    task = asyncio.create_task(object_index.index_file(file))
                    try:
                        while not task.done():
                            _ = await asyncio.wait({task}, timeout=progress_interval)
                            _ = bar.refresh()
                            refresh_status()
                        result = asdict(await task)
                    except Exception as exc:
                        result = {"semantics_file": str(file), "status": "failed",
                                  "error": f"{type(exc).__name__}: {exc}"}
                    finally:
                        if not task.done():
                            task.cancel()
                            _ = await asyncio.gather(task, return_exceptions=True)
                    result.update(corpus=str(folder), elapsed_seconds=perf_counter() - started)
                    results.append(result)
                    counts[result["status"]] += 1
                    if result["status"] == "failed":
                        _ = bar.write(f"FAILED: {file}\n{result['error']}")
                    _ = bar.set_postfix(file=str(file), **counts)
                    _ = bar.update(1)
                    refresh_status()
        except asyncio.CancelledError:
            state = "Cancelled"
            raise
        except BaseException:
            state = "Stopped after error"
            raise
        else:
            state = "Completed with failures" if counts["failed"] else "Completed"
        finally:
            refresh_status()
    return results


## Discover inputs and check the embedding model

Discovery validates all folders before contacting Ollama. Concurrent preflight checks every configured server using
`show`, so installed model tags resolve normally. If the server reports
capabilities, the model must support `embedding` on every host. No embeddings or index writes occur
in this step. On a preflight failure the HTTP client is closed; correct configuration
and rerun this cell before processing.

In [ ]:
# | notest
corpora = discover_semantics(INPUT_DIRS)
for input_dir, files in corpora:
    print(f"{input_dir}: {len(files)} unique semantics files")
print(f"Total unique semantics files: {sum(len(files) for _, files in corpora)}")
if "ollama_client" in globals():
    await ollama_client.aclose()
ollama_client = OllamaEmbeddingPool(hosts=OLLAMA_HOSTS,
    concurrency_per_server=SERVER_CONCURRENCY, headers=headers, request_timeout=REQUEST_TIMEOUT)
try:
    _ = await ollama_client.check_models(EMBED_MODEL)
except BaseException:
    await ollama_client.aclose()
    raise
print(f"Providers ready on all hosts: {EMBED_MODEL}")


## Manual corpus indexing

Only this cell opens or creates the persistent index. Every discovered file is
included. Unchanged published documents return `unchanged` without embedding requests.
Changed or interrupted documents reuse cached vectors for identical summaries.
The new generation becomes active only after the complete document is stored.
Failed or cancelled documents preserve their previously active catalog and vectors.

`results` is populated after each file and remains available after interruption.
The catalog and client stay open for inspection below. After an interruption, rerun
this cell to retry; run **Close resources** when finished. Rerunning this cell closes
the previous catalog handle before opening it again. Do not run another writer against
the same catalog. Inactive generations are retained by the core index for later maintenance.
The live concurrency panel appears below the document bar while processing.


In [ ]:
# | notest
if "object_index" in globals():
    object_index.close()
object_index = ObjectEmbeddingIndex(
    CATALOG_PATH, collection=None, llm_client=ollama_client,
    model=EMBED_MODEL, embedding_config=EMBEDDING_CONFIG,
    embedding_concurrency=EMBEDDING_CONCURRENCY,
)
results = []
concurrency_display = EmbeddingProgressDisplay(ollama_client,
    total=sum(len(files) for _, files in corpora))
batch_start = perf_counter()
try:
    _ = await embed_corpora(corpora, object_index, results,
        progress_interval=PROGRESS_INTERVAL, status_display=concurrency_display)
finally:
    batch_elapsed_seconds = perf_counter() - batch_start
    print(f"Batch wall time: {batch_elapsed_seconds:.3f}s; finished files: {len(results)}")

## Results and failures

`indexed` means a document generation was published; `unchanged` means the published
input and configuration already match; `failed` includes a diagnostic. `objects`
counts unique object records excluding the document. `embedded` counts newly computed
vectors and `reused` counts cached vectors (or all nonempty summaries for unchanged
files). `skipped` counts records with empty summaries, including the document if empty.
These counters include document-summary vectors where applicable. Timings describe
this run, including quick unchanged checks. Review failures, then rerun the batch.

In [ ]:
# | notest
results_df = pd.DataFrame(results)
display(results_df)
if not results_df.empty:
    print(results_df.groupby(["corpus", "status"]).size().to_string())
    counts = results_df.reindex(columns=["embedded", "reused", "skipped"]).fillna(0).sum()
    print("Vector/record counts for this run:")
    print(counts.to_string())
    failures = results_df.loc[results_df["status"] == "failed"]
    if not failures.empty:
        print("Failed files:")
        display(failures[["corpus", "semantics_file", "error"]])
print(f"Batch wall time: {batch_elapsed_seconds:.3f}s")
print("Per-server embedding requests:")
display(pd.DataFrame(ollama_client.server_snapshots()))


## Inspect published records

Inspect up to `PREVIEW_OUTPUTS` successful documents, including unchanged ones, through
the public catalog lookup API. Show the complete recursive hierarchy once, including
unembedded/ignored objects. Raw evidence and source locations remain available on each
record; this preview displays a short content excerpt. No model requests or source
writes occur. Set `PREVIEW_OUTPUTS = 0` to disable inspection.

In [ ]:
# | notest
def catalog_preview(index, document_id):
    rows = []
    def visit(identifier, depth=0):
        record = index.get_object(identifier)
        if record is None:
            raise ValueError(f"Missing published catalog record: {identifier}")
        rows.append({
            "id": record.id, "source_id": record.source_id, "parent_id": record.parent_id,
            "children_ids": record.children_ids, "depth": depth, "type": record.type,
            "object_path": record.object_path, "source_file": record.source_file,
            "line_start": record.attributes.get("line_start"),
            "line_end": record.attributes.get("line_end"),
            "summary_status": "embedded" if record.summary.strip() else "catalog only",
            "summary": record.summary, "content_preview": record.content[:200],
        })
        for child_id in record.children_ids:
            visit(child_id, depth + 1)
    visit(document_id)
    return rows


preview_documents = [row for row in results if row["status"] in {"indexed", "unchanged"}][:PREVIEW_OUTPUTS]
for row in preview_documents:
    print(f"Published objects: {row['semantics_file']}")
    preview_df = pd.DataFrame(catalog_preview(object_index, row["document_id"]))
    print(preview_df.groupby(["type", "summary_status"]).size().to_string())
    display(preview_df)

## Retrieve evidence for a future RAG answer

Use the same `object_index` and async provider pool as indexing. Search summaries,
then inspect the matched raw content, source location, ancestors and children before
passing evidence to an answer model. Edit `question` and `top_k` as needed; optional
`object_types` and `document_ids` filters restrict matches. This cell performs one
query embedding request and retrieval only; it does not generate an answer.

Run this before closing resources. No corpus files or existing vectors are rewritten.


In [ ]:
# | notest
question = '如何配置机器人焊接作业流程？'
hits = await object_index.search(question, top_k=5)
for hit in hits:
    record = hit.record
    print(f'\nDistance: {hit.distance:.4f} | {record.type} | {record.id}')
    print(record.semantics_file, record.object_path)
    print('Source:', record.source_file, record.attributes.get('line_start'), record.attributes.get('line_end'))
    print('Sections:', ' > '.join(record.section_headings))
    print('Summary:', hit.summary)
    print('Raw evidence:', record.content)
    print('Ancestors:', [(ancestor.type, ancestor.id) for ancestor in hit.ancestors])
    print('Children:', [(child.type, child.id, child.summary) for child in hit.children])
# Optional: restrict to a document or object type, e.g. object_types=['table_row', 'list_item'].


## Close resources and resume later

Close the SQLite catalog and Ollama HTTP client, including after an interrupted batch.
Run provider preflight again before another batch after closing resources. The completed
catalog and vector cache are the embedding resume state; no additional corpus checkpoint
is needed. Never reset the summary notebook's checkpoints to retry embedding.

The retrieval section above expands matching evidence through the catalog's
parent/child relationships. The [core object embedding notebook](01.core.dom.embedding.object.ipynb)
contains the reusable API and offline functional checks. This application notebook
does not generate RAG answers.


In [ ]:
# | notest
try:
    if "object_index" in globals():
        object_index.close()
finally:
    if "ollama_client" in globals():
        await ollama_client.aclose()

## Compare existing embedding indexes

This controlled comparison uses the two **explicit paths below**, independently of
`OBJECT_RAG_INDEX` and the indexing `INDEX_DIR`. Run this notebook’s configuration
cell and the comparison configuration below; the indexing cells are not required.
It uses the production `ObjectRAG`
retriever and answer prompt, category mappings, top-k 8, 24,000-character context
budget, and the configured answer model (temperature 0; 2,048 output tokens).
Each question is embedded once and the same query vector is reused for both stores.
The three general-operation questions from
[the RAG application](05.application.rag.object.ipynb) also receive generated answers. Five further queries check position copying, arc welding,
tracking, vision and a broad teaching-free-welding request.

The seven narrow questions have **predeclared source-text targets**. Target hit@8,
first-target rank and target-in-context measure whether that specific evidence was
found; they are diagnostic checks, not human-labelled precision/recall or proof of
answer correctness. The broad welding request is unscored. Inspect the cited source
excerpts and generated answers as well as the numbers. A lower cosine distance or
an `included` example status alone does not establish a better answer.

Catalogs are opened read-only. Chroma is queried through disposable copies because
its persistent client can write housekeeping files even during retrieval. No
indexing or summary/checkpoint operation is performed. Temporary Chroma clients
are stopped and copies removed, including on failure or cancellation.

**Observed assessment (2026-10-27, Asia/Shanghai): prefer the timestamped store for
the coding queries tested here.** It retrieves the declared target more often and
ranks the assignment evidence more directly. The base store has substantially
more complete vector coverage. The detailed assessment below distinguishes these
findings; the configured `INDEX_DIR` remains the user's existing selection.

Run the retrieval cell, then the three answer cells. Answer generation replays the
frozen retrieval results through the unchanged `ObjectRAG.answer` method, so each
answer is assessed against exactly the evidence shown in this comparison. It sends
those excerpts to `RAG_HOST` (`http://172.27.74.16:11434` in this recorded run), which
was explicitly authorized for this comparison. Saved outputs record the measured
run; changing models, services, mappings or either store can change later results.


In [ ]:
# | notest
from IPython.display import Markdown
from ribosome.rag.object import ObjectRAG

# Reuse the embedding configuration without changing the indexing destination.
HOSTS = sorted(set(OLLAMA_HOSTS))
RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
RAG_HOST = os.getenv("OLLAMA_RAG_HOST") or os.getenv("OLLAMA_HOST", "http://localhost:11434")
MAX_CONTEXT_CHARS = 24000
MAX_OUTPUT_TOKENS = 2048


In [2]:
# | notest
import html
import json
import re
import shutil
import tempfile
from collections import Counter
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo

import chromadb
from chromadb.config import Settings
from ollama import AsyncClient
from ribosome.rag.object import canonical

COMPARISON_INDEXES = {
    'base': PROJ_ROOT / 'db/object_embeddings/qwen3-embedding-8b',
    'timestamped': PROJ_ROOT / 'db/object_embeddings/qwen3-embedding-8b-20261026-090625-217683',
}
COMPARISON_CASES = [
    dict(id='assignment', category='general_operation',
         question='如何进行赋值运算，将参数2的值赋给参数1?',
         target=r'赋值运算[，,]?\s*将参数2的值赋给参数1', generate=True),
    dict(id='user_frame', category='general_operation',
         question='如何设置用户坐标号?', target=r'SET\s+UF\b', generate=True),
    dict(id='interrupt_position', category='general_operation',
         question='获取中断发生瞬间的机器人当前位置的规划值或实际值赋值给位置变量PR?',
         target=r'GET\s+IRQP\b', generate=True),
    dict(id='position_copy', category='general_operation',
         question='如何把位置变量PR2的值赋给PR1？', target=r'SET\s+PR1\s+PR2\b'),
    dict(id='arc_start', category='arc_welding',
         question='如何使用ARCON指令开始焊接？', target=r'\bARCON\b'),
    dict(id='tracking_speed', category='tracking',
         question='如何读取跟随运动中的实时速度？', target=r'\bFL_GETV\b'),
    dict(id='camera_trigger', category='vision',
         question='如何给相机发送拍照指令？', target=r'给相机发送拍照指令'),
    dict(id='teaching_free', category='teaching_free_welding',
         question='如何编写免示教焊接程序？', target=None),
]
COMPARISON_TOP_K = 8

def comparison_text(text):
    return ' '.join(html.unescape(re.sub(r'<[^>]*>', ' ', text)).split())

class SharedQueryEmbedder:
    """Cache only query embeddings in memory; never update either catalog."""
    def __init__(self, client):
        self.client, self.cache = client, {}

    async def embed(self, *, model, input):
        key = (model, input)
        if key not in self.cache:
            self.cache[key] = await self.client.embed(model=model, input=input)
        return self.cache[key]

def comparison_catalog_stats(app, label):
    rows = app.connection.execute(
        "SELECT o.record,o.vector_id FROM objects o JOIN active_documents a "
        "ON a.generation_id=o.generation_id"
    )
    counts, vectors = Counter(), Counter()
    for raw, vector_id in rows:
        kind = json.loads(raw)['type']
        counts[kind] += 1
        vectors[kind] += bool(vector_id)
    categories = {}
    # Mapping coverage is determined by document identity, not vector count.
    for raw, in app.connection.execute(
        "SELECT o.record FROM objects o JOIN active_documents a "
        "ON a.generation_id=o.generation_id WHERE json_extract(o.record,'$.type')='document'"
    ):
        record = json.loads(raw)
        key, profile = canonical(record['semantics_file'])
        for entry in app.mapping:
            if entry['key'] == key and (not entry['profile'] or entry['profile'] == profile):
                categories.setdefault(entry['category'], set()).add((key, profile))
    return dict(index=label, documents=counts['document'],
                mapped_categories=len(categories), objects=sum(counts.values()),
                table_objects=counts['table'], table_vectors=vectors['table'],
                row_objects=counts['table_row'], row_vectors=vectors['table_row'],
                dimension=app.dimension,
                identity=json.loads(dict(app.connection.execute('SELECT key,value FROM settings'))['identity']),
                active_generations=list(app.connection.execute('SELECT * FROM active_documents ORDER BY document_id')))

async def compare_object_indexes():
    """Run both stores with identical settings; preserve all results for inspection."""
    auth = {'Authorization': 'Bearer '+os.environ['OLLAMA_API_KEY']} if os.getenv('OLLAMA_API_KEY') else {}
    embed_client = AsyncClient(host=HOSTS[0], headers=auth, timeout=REQUEST_TIMEOUT)
    shared_embedder = SharedQueryEmbedder(embed_client)
    results, stats, clients, apps = {}, [], [], []
    report = dict(run_at=datetime.now(ZoneInfo('Asia/Shanghai')).isoformat(),
                  settings=dict(embed_model=EMBED_MODEL, hosts=HOSTS, rag_model=RAG_MODEL,
                                rag_host=RAG_HOST, top_k=COMPARISON_TOP_K,
                                max_context_chars=MAX_CONTEXT_CHARS,
                                max_output_tokens=MAX_OUTPUT_TOKENS, temperature=0),
                  stats=stats, results=results)
    try:
        with tempfile.TemporaryDirectory(prefix='object-rag-comparison-') as temporary:
            try:
                for label, directory in COMPARISON_INDEXES.items():
                    if not (directory/'catalog.sqlite3').is_file() or not (directory/'chroma').is_dir():
                        raise FileNotFoundError(directory)
                    destination = Path(temporary)/label
                    shutil.copytree(directory/'chroma', destination)
                    client = chromadb.PersistentClient(
                        path=str(destination), settings=Settings(anonymized_telemetry=False))
                    clients.append(client)
                    collection = client.get_collection('object_summaries', embedding_function=None)
                    candidate_app = ObjectRAG(
                        directory, collection, shared_embedder,
                        mapping_dir=PROJ_ROOT/'assets/coding_categories', embed_model=EMBED_MODEL,
                        embedding_config=dict(provider='ollama-pool', hosts=HOSTS),
                        rag_model=RAG_MODEL, max_context_chars=MAX_CONTEXT_CHARS,
                        max_output_tokens=MAX_OUTPUT_TOKENS)
                    apps.append(candidate_app)
                    stats.append(comparison_catalog_stats(candidate_app, label))
                for case in COMPARISON_CASES:
                    results[case['id']] = {}
                    for (label, _), candidate_app in zip(COMPARISON_INDEXES.items(), apps):
                        print(f"{case['id']}: {label}", flush=True)
                        started = perf_counter()
                        try:
                            result = await candidate_app.retrieve(case['question'], case['category'], top_k=COMPARISON_TOP_K)
                            result = dict(result)
                        except Exception as error:
                            result = dict(candidate_app.last_result or {})
                            result['error'] = f'{type(error).__name__}: {error}'
                        result['elapsed_seconds'] = perf_counter()-started
                        results[case['id']][label] = result
            finally:
                for candidate_app in apps:
                    candidate_app.close()
                # PersistentClient has no public close API in the installed Chroma.
                for client in clients:
                    client._system.stop()
    finally:
        await embed_client._client.aclose()
    report['query_embedding_requests'] = len(shared_embedder.cache)
    return report


In [3]:
# | notest
comparison = await compare_object_indexes()


assignment: base


assignment: timestamped


user_frame: base


user_frame: timestamped


interrupt_position: base


...

tracking_speed: timestamped


camera_trigger: base


camera_trigger: timestamped


teaching_free: base


teaching_free: timestamped


In [3]:
# | notest
from copy import deepcopy

class ComparisonEvidenceReplay(ObjectRAG):
    """Use the unchanged production answer method on the frozen retrieval result."""
    def __init__(self, evidence, generator):
        self.evidence = evidence
        self.generator = generator
        self.rag_model = RAG_MODEL
        self.max_output_tokens = MAX_OUTPUT_TOKENS
        self.last_result = None

    async def retrieve(self, question, category, *, top_k=8):
        if (question, category, top_k) != (
            self.evidence['question'], self.evidence['category'], COMPARISON_TOP_K
        ):
            raise ValueError('Replay requires the same question, category and top-k')
        self.last_result = deepcopy(self.evidence)
        return self.last_result

async def compare_saved_answers(report, case_id):
    case = next(case for case in COMPARISON_CASES if case['id'] == case_id)
    auth = {'Authorization': 'Bearer '+os.environ['OLLAMA_API_KEY']} if os.getenv('OLLAMA_API_KEY') else {}
    client = AsyncClient(host=RAG_HOST, headers=auth, timeout=REQUEST_TIMEOUT)
    try:
        for label, evidence in report['results'][case_id].items():
            print(f"Answer {case_id}: {label}", flush=True)
            app_replay = ComparisonEvidenceReplay(evidence, client)
            started = perf_counter()
            try:
                result = await app_replay.answer(case['question'], case['category'], top_k=COMPARISON_TOP_K)
            except Exception as error:
                result = dict(app_replay.last_result or evidence)
                result['error'] = f'{type(error).__name__}: {error}'
            result['answer_elapsed_seconds'] = perf_counter()-started
            report['results'][case_id][label] = result
    finally:
        await client._client.aclose()


In [5]:
# | notest
await compare_saved_answers(comparison, 'assignment')


Answer assignment: base


Answer assignment: timestamped


In [6]:
# | notest
await compare_saved_answers(comparison, 'user_frame')


Answer user_frame: base


Answer user_frame: timestamped


In [7]:
# | notest
await compare_saved_answers(comparison, 'interrupt_position')


Answer interrupt_position: base


Answer interrupt_position: timestamped


In [8]:
# | notest
display(pd.DataFrame(comparison['stats']).drop(columns=['identity', 'active_generations']))
comparison_rows, evidence_rows = [], []
for case in COMPARISON_CASES:
    for label, result in comparison['results'][case['id']].items():
        pattern = case['target']
        matching = [rank for rank, hit in enumerate(result.get('hits', []), 1)
                    if pattern and re.search(pattern, comparison_text(hit['record']['content']))]
        context_match = bool(re.search(pattern, comparison_text(result.get('context', '')))) if pattern else None
        comparison_rows.append(dict(
            query=case['id'], index=label, hits=len(result.get('hits', [])),
            target_hit_at_8=bool(matching) if pattern else None,
            first_target_rank=matching[0] if matching else None,
            target_in_context=context_match, context_chars=len(result.get('context', '')),
            omissions=len(result.get('omissions', [])),
            missing_manuals=len(result.get('coverage', {}).get('missing', [])),
            example_status=result.get('example_status', 'no evidence' if 'answer' in result else 'not generated'),
            error=result.get('error', '')))
        # Keep all eight exact citations for the notebook's current assignment query.
        # For other queries show the top hit and first target hit when different.
        ranks = range(1, len(result.get('hits', []))+1) if case['id']=='assignment' else sorted({1, *matching[:1]})
        for rank in ranks:
            if rank > len(result.get('hits', [])): continue
            hit = result['hits'][rank-1]
            record, manual = hit['record'], hit['manual']
            location = record['attributes']
            evidence_rows.append(dict(
                query=case['id'], index=label, rank=rank, distance=round(hit['distance'], 6),
                corpus=manual['profile'], revision=manual['revision'],
                manual=Path(record['source_file']).name,
                lines=f"{location.get('line_start')}-{location.get('line_end')}",
                object_id=record['id'],
                source_excerpt=comparison_text(record['content'])[:360],
                stored_summary=record.get('summary', '')[:300]))
comparison_metrics = pd.DataFrame(comparison_rows)
comparison_evidence = pd.DataFrame(evidence_rows)
display(comparison_metrics)
display(comparison_evidence)
scored = comparison_metrics[comparison_metrics['query'] != 'teaching_free'].copy()
scored['reciprocal_rank'] = 1/scored['first_target_rank']
scored['reciprocal_rank'] = scored['reciprocal_rank'].fillna(0)
display(scored.groupby('index').agg(
    target_hit_rate_at_8=('target_hit_at_8', 'mean'),
    mean_reciprocal_target_rank=('reciprocal_rank', 'mean'),
    target_context_rate=('target_in_context', 'mean'),
    omitted_objects=('omissions', 'sum')))
print('Run:', comparison['run_at'], '\nSettings:', comparison['settings'],
      '\nShared query embedding requests:', comparison['query_embedding_requests'])


,query,index,hits,target_hit_at_8,first_target_rank,target_in_context,context_chars,omissions,missing_manuals,example_status,error
0,assignment,base,8,True,2.0,True,22373,0,0,included,
1,assignment,timestamped,8,True,1.0,True,21378,0,0,included,
2,user_frame,base,8,False,NaN,False,20283,0,0,included,
3,user_frame,timestamped,0,False,NaN,False,0,0,0,no evidence,
4,interrupt_position,base,8,True,1.0,True,23756,0,0,included,
5,interrupt_position,timestamped,8,True,1.0,True,22258,0,0,included,
6,position_copy,base,8,False,NaN,True,21504,0,0,not generated,
7,position_copy,timestamped,8,True,2.0,True,20387,0,0,not generated,
8,arc_start,base,8,True,1.0,True,19603,0,0,not generated,
9,arc_start,timestamped,8,True,1.0,True,22835,0,0,not generated,


,query,index,rank,distance,corpus,revision,manual,lines,object_id,source_excerpt,stored_summary
0,assignment,base,1,0.263820,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,7266-7270,41f9342677ec4cdcb51e5d15d9eab8ec10a7dc2d0dea3b...,位置变量 P 功能 赋值运算，将参数2的位置变量赋值给参数1中的位置变量。,位置变量 P 执行赋值运算，将参数 2 的位置变量值赋给参数 1。
1,assignment,base,2,0.299474,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,14855-14859,c3cd1ed93539768efd46668d32230f4083cb2d9e4e2053...,整型 SET 功能 赋值运算，将参数2的值赋给参数1,该单元格行描述**整型 SET**指令的功能：**赋值运算**，即**将参数2的值赋给参数1**。
2,assignment,base,3,0.309205,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,15979-15983,e060e94322125b1850c5e8d1ba32d1719e1c622c6846bf...,位置变量1 PRI_PR 功能 赋值运算，把P变量赋值给PRI变量，I为整形变量。,位置变量1（PRI_PR）：执行赋值运算，将P变量赋值给PRI变量（I为整形变量）。
3,assignment,base,4,0.322614,5.0,V5.0 / C-1,SX322001《新松机器人通用操作手册》(C-1).md,10940-10942,15b6d66bbed58f4e70560009aa351e1b0cc5b7205a4506...,S[]：字符型变量，能直接赋值,SET指令的“参数2”说明：S[]为字符型变量，支持直接赋值。
4,assignment,base,5,0.324595,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,15629-15633,960f5a34ed641cce9ec615f914263c66b3adb9898f1ca1...,位置变量1 常数 功能 赋值运算，将实型常数赋值给位置变量的分量,该条目描述对“位置变量1”的分量进行赋值运算：将实型常数赋给该位置变量的指定分量（功能为“赋...
5,assignment,base,6,0.327789,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,6667-6671,deaa56c9b838b88e2b5d2e4d0481f92df69b4b07b09c25...,整型 DIV 功能 除法运算，参数1与参数2相除，结果存在参数1中。,整型 DIV：执行除法运算，将参数1除以参数2，结果存入参数1。
6,assignment,base,7,0.333647,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,15279-15283,14ac6c058f8342fbcdcec0085bd99ca715865400b27296...,字节型 SET 功能 赋值运算，将参数2的值赋给字节型参数1,**字节型 SET**：执行赋值运算，将参数2的值赋给字节型参数1。
7,assignment,base,8,0.338058,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,6557-6561,860e62b7b8ba7f903e1dc68092d623340f003047856201...,整型 SET 功能 赋值运算，将参数2的值赋给参数1。,整型 SET 的功能为赋值运算，即将参数2的值赋给参数1。
8,assignment,timestamped,1,0.234462,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,14855-14859,c3cd1ed93539768efd46668d32230f4083cb2d9e4e2053...,整型 SET 功能 赋值运算，将参数2的值赋给参数1,整型 SET：执行赋值运算，将参数 2 的值赋给参数 1。
9,assignment,timestamped,2,0.309205,4.0,V5.0 / B-5,SX322001《新松机器人通用操作手册》(B-5).md,15979-15983,e060e94322125b1850c5e8d1ba32d1719e1c622c6846bf...,位置变量1 PRI_PR 功能 赋值运算，把P变量赋值给PRI变量，I为整形变量。,位置变量1（PRI_PR）：执行赋值运算，将P变量赋值给PRI变量（I为整形变量）。


,target_hit_rate_at_8,mean_reciprocal_target_rank,target_context_rate,omitted_objects
index,,,,
base,0.714286,0.642857,0.857143,0
timestamped,0.857143,0.785714,0.857143,0


In [9]:
# | notest
for case in COMPARISON_CASES:
    if not case.get('generate'): continue
    display(Markdown(f"#### {case['id']}: {case['question']}"))
    for label, result in comparison['results'][case['id']].items():
        display(Markdown(f"**{label} — {result.get('example_status', 'generation failed')}**\n\n"
                         + result.get('answer', result.get('error', 'No answer'))))


#### assignment: 如何进行赋值运算，将参数2的值赋给参数1?

**base — included**

在新松机器人通用操作手册中，整型 SET 指令用于赋值运算，将参数2的值赋给参数1。格式为 SET <参数1> <参数2>，其中参数1可以是整型用户变量 I<变量号>（范围1-520）、8位组IO输出 OG#<变量号>（范围1-8）或16位组IO输出 OGH#<变量号>（范围1-64）；参数2可以是常数（范围-99999~99999）、整型用户变量、IO输入或实型变量。例如 SET I1 I2 表示将 I2 的值赋给 I1。

### Example — general_operation

在机器人程序中，需要将整型变量 I2 中存储的计数值复制到 I1 中，以便后续对 I1 进行累加运算。

**Environment and prerequisites:** 新松机器人控制器，整型用户变量 I1~I520 可用，实型变量 R1~R520 可用。I2 已预先赋值为 200，I3 已预先赋值为 50。

**Expected result:** 执行 SET I1 I2 后，I1 的值变为 200（与 I2 相同）。随后执行 ADD I1 I3 后，I1 的值变为 250。

**Manual:** SX322001《新松机器人通用操作手册》(B-5) [E2] [E6] [E8]

```text
SET I2=200
SET I1 I2
ADD I1 I3
```

**timestamped — included**

在新松机器人编程语言中，整型赋值运算使用 SET 指令，格式为 SET <参数1> <参数2>，功能是将参数2的值赋给参数1。参数1可以是整型用户变量 I<变量号>（范围1-520）、8位组IO输出 OG#<变量号>（范围1-8）或16位组IO输出 OGH#<变量号>（范围1-64）。参数2可以是常数（范围-99999~99999）、整型用户变量 I<变量号>、8位组IO输入 IG#<变量号>、16位组IO输入 IGH#<变量号>、32位组IO输入 IGFA#<变量号>或实型用户变量 R<变量号>（小数部分舍去）。

### Example — general_operation

在机器人程序中，需要将整型变量 I2 中已存储的计数值（256）复制到整型变量 I1 中，以便后续将 I1 的值传递给 I3 用于控制逻辑判断。

**Environment and prerequisites:** 整型用户变量 I1、I2、I3（范围1-520）；I2 初始值为 256；I1 和 I3 初始值未定义。

**Expected result:** 执行 SET I1 I2 后，I1 的值变为 256；随后执行 SET I3 I1 后，I3 的值也变为 256，验证赋值结果正确传递。

**Manual:** SX322001《新松机器人通用操作手册》(B-5) V5.0 / B-5 [E1] [E6]

```text
SET I2=256
SET I1 I2
SET I3 I1
```

#### user_frame: 如何设置用户坐标号?

**base — included**

根据新松机器人通用操作手册（V5.0/B-5），设置用户坐标号的操作路径为：主菜单「坐标」→ 二级菜单「用户坐标」→ 三级菜单「设定」，需要高级用户权限。E8（V5.0/C-1）进一步说明用户坐标系文件号变更时，用户坐标系图标上的数字会相应变更，并提醒用户坐标系号的改变在偏移指令等情况下会影响机器人程序运动轨迹，需在手动或自动执行程序前确认。

### Example — general_operation

操作员需要在示教器上将当前作业使用的用户坐标系号（文件号）切换为另一个已标定的坐标系，以便后续程序在正确的用户坐标系下运行。

**Environment and prerequisites:** 前提条件：已登录高级用户权限；机器人处于手动模式；至少存在一个已标定的用户坐标系文件。变量：目标用户坐标系文件号（由操作员指定，如 U2）。初始状态：当前用户坐标系图标显示为 U1。

**Expected result:** 进入「坐标 → 用户坐标 → 设定」菜单后，将用户坐标系号修改为目标文件号（如 U2），确认后示教器上用户坐标系图标上的数字由 U1 变更为 U2，后续运动指令将基于 U2 坐标系执行。

**Manual:** SX322001《新松机器人通用操作手册》V5.0 / B-5 [E1] [E2] [E4] [E6] [E8]

```text
用户等级: 高级用户
坐标 → 用户坐标 → 设定
用户坐标系图标上的数字由 U1 变更为 U2
```

**timestamped — generation failed**

No eligible evidence is indexed for this category or fits the context budget.

#### interrupt_position: 获取中断发生瞬间的机器人当前位置的规划值或实际值赋值给位置变量PR?

**base — included**

The GET IRQP instruction (documented in the 新松 robot general operation manual) captures the robot's current position—either the planned value (参数1=1) or the actual value (参数1=2)—at the exact moment an interrupt occurs, and assigns it to a specified PR position variable. The instruction must be placed inside the interrupt program. The format is: IRQP<1|2> LEV=<interrupt_level> PR<position_var_number>. In the example, IRQP1 LEV=1 PR10 retrieves the planned position at interrupt level 1 and stores it in PR10.

### Example — general_operation

A robot is executing an automatic welding program when an external safety sensor triggers an interrupt (level 1). The interrupt program must immediately capture the robot's planned position at that instant for post-incident analysis, then convert the stored position to Cartesian (orthogonal) form for reporting.

**Environment and prerequisites:** Prerequisites: robot is in automatic mode executing a program; interrupt level 1 is configured and mapped to an external I/O signal; PR10 is an available global position variable (valid range 1–5630). The instruction must reside within the interrupt subroutine, not the main program. No user-supplied placeholders are needed; all values are concrete constants from the manual example.

**Expected result:** After the interrupt fires, PR10 holds the robot's planned Cartesian position (X, Y, Z, RX, RY, RZ) frozen at the instant of interruption. The subsequent CONVERT instruction re-expresses PR10 in orthogonal (Cartesian) coordinate form (COORD=2), making the data ready for diagnostic logging.

**Manual:** SX322001《新松机器人通用操作手册》(B-5), V5.0 / B-5 [E1]

```text
SET VEL 0
IRQP1 LEV=1 PR10
CONVERT PR10 COORD=2
```

**timestamped — included**

根据手册E1，GET IRQP指令用于获取中断发生瞬间的机器人当前位置的规划值或实际值并赋值给位置变量PR。格式为 IRQP<参数1> LEV=<参数2> P<参数3>，其中参数1为1（规划值）或2（实际值），参数2为中断等级（需与调用作业的中断等级一致），参数3为位置变量号（范围1-5630）。该指令必须在中断程序中使用。示例中，机器人在自动执行焊接任务时触发紧急停止中断（等级1），中断程序通过 IRQP2 LEV=1 PR10 将中断瞬间的实际位置捕获到PR10，随后用 SET PR11 PR10 将该位置复制到PR11用于安全日志记录。

### Example — general_operation

机器人在自动模式下执行焊接作业，运行过程中触发紧急停止中断（中断等级1）。中断程序需要捕获中断发生瞬间机器人的实际位置，以便后续安全分析和故障诊断。

**Environment and prerequisites:** 机器人处于自动运行模式，正在执行焊接程序；紧急停止按钮被按下触发等级1中断；中断程序已预先编写；位置变量PR10用于存储中断瞬间位置，PR11用于备份日志；速度已设为80%。

**Expected result:** 中断触发后，IRQP2 LEV=1 PR10 将中断瞬间机器人的实际位置（6个分量：X、Y、Z、RX、RY、RZ）写入PR10；随后 SET PR11 PR10 将该位置完整复制到PR11，供安全日志模块读取。

**Manual:** SX322001《新松机器人通用操作手册》(B-5) V5.0 / B-5 [E1] [E4]

```text
SET VEL 80
IRQP2 LEV=1 PR10
SET PR11 PR10
```

### Assessment of the recorded run

`base` = `db/object_embeddings/qwen3-embedding-8b`;
`timestamped` = `db/object_embeddings/qwen3-embedding-8b-20261026-090625-217683`.
The run used `qwen3-embedding:8b` for queries and `qwen3.8:27b` for answers.
All sixteen retrievals used top-k 8 and the same context budget; there were no
context omissions or missing mapped manuals. One of the six answer calls had empty
retrieval context and therefore made no generation request.

| Check | base | timestamped | Interpretation |
|---|---:|---:|---|
| Active documents / mapped categories | 38 / 18 | 38 / 18 | Same manual coverage |
| Active objects | 86,811 | 86,811 | Same object IDs in both stores |
| Table vectors / table objects | 1,350 / 1,352 | 652 / 1,352 | Base has broader whole-table coverage |
| Row vectors / row objects | 15,503 / 15,505 | 15,316 / 15,505 | Base also covers more rows |
| Declared source target found in top 8 | 5/7 (71.4%) | 6/7 (85.7%) | Timestamped wins this diagnostic |
| Mean reciprocal first-target rank | 0.643 | 0.786 | Timestamped ranks targets earlier |
| Target present in final context | 6/7 (85.7%) | 6/7 (85.7%) | Parent-table expansion closes the context gap |
| First target rank: assignment | 2 | 1 | Timestamped finds the requested integer SET function first |
| First target rank: position copy | Absent from hits | 2 | Base still recovers the syntax through parent-table context |

**For the RAG application's assignment question, timestamped is better.** Its
first hit is the integer `SET` function in the B-5 manual, lines 14855–14859
(distance 0.234462). Base puts the position-variable variant at rank 1
(lines 7266–7270, distance 0.263820), then the intended integer function at rank 2.
Both generated answers correctly use `SET I1 I2`. Timestamped gives a focused
three-SET example with one manual revision; base adds a documented `ADD` follow-up.
The recommendation rests on evidence relevance and the examples, rather than the
distance comparison alone.

**The other examples expose limits in both pipelines.** Both rank `GET IRQP` first
for interrupt-position capture (B-5, lines 14717–14721) and include its surrounding
syntax. Base's answer is in English despite the Chinese question, and its E1-only
citation does not cover the `CONVERT` follow-up, which appears in a different
retrieved table. Timestamped answers in Chinese and cites the copy operation,
but its claims about an emergency-stop trigger and six Cartesian components are
not established by the supplied IRQP evidence. Neither generated example was
executed on a controller. An `included` status only passes the application's
structural validator; it is not a correctness score.

For the original user-coordinate query, base retrieved menu and permission rows
and produced menu steps inside a code fence, mixing B-5 and C-1 evidence. That is
not a grounded robot-program example, despite `example_status='included'`.
Timestamped initially returned no hits and abstained. A follow-up with **one fresh
query embedding**, replayed three times per store, returned eight hits each time
for both stores. Exact cosine ranking of that follow-up vector over the same
eligible candidates (6,921 base; 6,332 timestamped) placed timestamped's `SET UF`
function row at rank 4 (B-5, lines 14473–14477). The initial empty result should
therefore not be interpreted as permanent missing manual coverage. The fresh-vector
check is separate from the scored run and does not replace its results.

For the broad teaching-free-welding question, both stores mostly return cover and
contents tables, rather than programming instructions. It is unscored because no
single target was declared. The shared table/outer-row candidate restriction leaves
useful prose and list content outside retrieval; changing indexes alone does not
solve this query.

**Recommendation:** use timestamped for the tested assignment and narrow coding
queries, and retain base when broad table-vector coverage is the priority. This is
a small diagnostic sample, with three generated-answer pairs, rather than evidence
of universal superiority. The indexes have the same embedding fingerprint and
dimension, but different stored summaries: a read-only audit found changes in
15,359 of 15,505 row summaries. Their row source content and object IDs match.
This comparison therefore evaluates the actual stored summaries, vector coverage,
ranking and downstream answers together, not two different embedding models.
